<a href="https://colab.research.google.com/github/sam-wahid/vlm-llm-segmentation/blob/main/dinov2benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q torch torchvision timm
!pip install -q scikit-learn tqdm matplotlib

In [ ]:
import time
import torch
import numpy as np

from tqdm import tqdm

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cpu


In [ ]:
dinov2_model = torch.hub.load(
    'facebookresearch/dinov2',
    'dinov2_vits14'
)

dinov2_model = dinov2_model.to(device)
dinov2_model.eval()

print("DINOv2 Loaded")

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 344MB/s]


DINOv2 Loaded


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor()
])

dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

loader = DataLoader(dataset, batch_size=32, shuffle=False)

print("Total Images:", len(dataset))
print("Classes:", dataset.classes)

100%|██████████| 26.4M/26.4M [00:02<00:00, 12.8MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 211kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.83MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 18.7MB/s]

Total Images: 10000
Classes: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


In [ ]:
def benchmark_dinov2(model, loader):

    embeddings = []
    labels_all = []

    total_time = 0
    total_images = 0

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    with torch.no_grad():

        for images, labels in tqdm(loader):

            images = images.to(device)

            start = time.time()

            features = model(images)

            end = time.time()

            total_time += (end - start)
            total_images += images.size(0)

            embeddings.append(features.cpu().numpy())
            labels_all.append(labels.numpy())

    embeddings = np.concatenate(embeddings, axis=0)
    labels_all = np.concatenate(labels_all, axis=0)

    fps = total_images / total_time
    latency = total_time / total_images

    gpu_memory = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else 0

    return {
        "embeddings": embeddings,
        "labels": labels_all,
        "fps": fps,
        "latency": latency,
        "gpu_memory": gpu_memory
    }

In [ ]:
dinov2_results = benchmark_dinov2(dinov2_model, loader)

  6%|▋         | 20/313 [02:38<37:07,  7.60s/it]

In [ ]:
X = dinov2_results["embeddings"]
y = dinov2_results["labels"]

split = int(0.8 * len(X))

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

pred = knn.predict(X_test)

acc = accuracy_score(y_test, pred)

In [ ]:
print("\n===== DINOv2 RESULTS =====")

print("Accuracy      :", round(acc * 100, 2), "%")
print("FPS           :", round(dinov2_results["fps"], 2))
print("Latency       :", round(dinov2_results["latency"], 5))
print("GPU Memory GB :", round(dinov2_results["gpu_memory"], 2))